# 3. Ablations, read against the noise

An ablation delta is meaningless without knowing how far two runs of the *same* configuration drift apart. The seed study is therefore run first and every delta is divided by `sqrt(2) * sd` before it is interpreted.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, torch
torch.set_num_threads(2)
pd.set_option("display.width", 200)
import matplotlib.pyplot as plt

# Notebooks run with cwd=notebooks/, so the figure and table roots have to be
# repointed at the repository's results/ tree. Without this every figure lands in
# notebooks/results/figures/ and the committed figures silently never update.
import saqa.viz as _viz, saqa.report as _report
_viz.FIGURES = ROOT / "results" / "figures"
_viz.TABLES = _report.TABLES = ROOT / "results" / "tables"
tables = _viz.TABLES
tables = pathlib.Path('../results/tables')
var = pd.read_csv(tables / 'seed_variance.csv') if (tables / 'seed_variance.csv').exists() else None
var.round(4) if var is not None else 'run `make ablate` first'

,metric,mean,sd,min,max,range,noise_scale,n_runs
0,spearman,0.3585,0.0852,0.2634,0.4280,0.1645,0.1205,3
1,kendall_tau,0.2481,0.0570,0.1849,0.2955,0.1106,0.0806,3
2,relative_l2,0.2456,0.0076,0.2368,0.2502,0.0134,0.0108,3
3,mae,0.1466,0.0056,0.1415,0.1526,0.0111,0.0080,3
4,coverage,0.9134,0.0232,0.8960,0.9398,0.0438,0.0328,3
5,mean_width,0.5997,0.0127,0.5859,0.6109,0.0250,0.0180,3
6,aurc,0.1411,0.0136,0.1269,0.1540,0.0271,0.0192,3
7,error_auroc,0.5180,0.0365,0.4761,0.5431,0.0670,0.0516,3


In [2]:
abl = pd.read_csv(tables / 'ablation_components.csv') if (tables / 'ablation_components.csv').exists() else None
cols = ['variant', 'change', 'spearman', 'spearman_delta', 'spearman_ratio_to_noise', 'spearman_verdict']
abl[[c for c in cols if c in abl.columns]].round(4) if abl is not None else None

,variant,change,spearman,spearman_delta,spearman_ratio_to_noise,spearman_verdict
0,full,-,0.3599,0.0000,0.0000,inside noise
1,partitions_uniform,model.partitions=uniform,0.4588,0.0989,0.8209,inside noise
2,partitions_identity,model.partitions=identity,0.4022,0.0423,0.3512,inside noise
3,no_edge_importance,model.edge_importance=False,0.3565,-0.0034,0.0280,inside noise
4,dense_temporal,model.separable=False,0.3510,-0.0089,0.0735,inside noise
5,temporal_kernel_3,model.temporal_kernel=3,0.3828,0.0229,0.1904,inside noise
6,head_regression,model.head=regression,0.3903,0.0304,0.2525,inside noise
7,norm_group,model.norm=group,0.0746,-0.2853,2.3676,survives
8,uncertainty_heteroscedastic,model.uncertainty=heteroscedastic,0.3533,-0.0066,0.0548,inside noise


## The normalisation choice is load-bearing

The first version of this model used GroupNorm everywhere, for batch-size independence. It could not fit its own training set. GroupNorm removes each sample's per-channel scale at every layer, and the readout is a global average pool over (T, V) -- which reads precisely that scale. This is measured below, not asserted.

In [3]:
from saqa.config import load_config
from saqa.pipelines import make_splits, run_single
from saqa.metrics import spearman
from saqa.engine import predict
cfg = load_config('../configs/base.yaml', ['data.num_sequences=300', 'optim.epochs=4', 'run.out_dir=../results/runs_nb'])
sp = make_splits(cfg)
for norm in ('batch', 'group'):
    c = load_config('../configs/base.yaml', ['data.num_sequences=300', 'optim.epochs=4', f'model.norm={norm}', 'run.out_dir=../results/runs_nb'])
    r = run_single(c, name=f'nb_norm_{norm}', splits=sp, save=False, verbose=False)
    tr = spearman(sp.train.quality, predict(r.model, sp.train.coords)['score'])
    print(f'{norm:6s} train rho {tr:+.4f}   test rho {r.metrics["spearman"]:+.4f}')

batch  train rho +0.2137   test rho +0.2107


group  train rho +0.0851   test rho +0.0802


## Splitting regimes: the leakage is measured, not assumed

A random split lets the same subject and the same defect combination appear on both sides. The gap between it and the cross-subject split is the leakage estimate.

In [4]:
p = tables / 'split_comparison.csv'
pd.read_csv(p)[['split', 'spearman', 'kendall_tau', 'relative_l2', 'mae']].round(4) if p.exists() else 'run `make splits` first'

,split,spearman,kendall_tau,relative_l2,mae
0,random,0.3333,0.2344,0.2660,0.1556
1,subject,0.3842,0.2641,0.2498,0.1526
2,combination,0.1679,0.1149,0.5936,0.2648
